# Finansal Varl?klar Aras? ?li?kilerin Sosyal A? Analizi

Bu notebook, sosyal a? analizi proje teslimi i?in haz?rlanm??t?r. ?al??mada finansal varl?klar d???m, g?nl?k getiri serileri aras?ndaki g??l? Pearson korelasyon ili?kileri ise kenar olarak modellenir. Kod ak??? mevcut proje klas?r?ndeki dosya yap?s?n? korur ve analiz ad?mlar?n? s?ral? bi?imde tekrar ?retir.

## 1. Kurulum ve K?t?phaneler

Bu b?l?mde analizde kullan?lacak Python k?t?phaneleri, veri yollar? ve temel parametreler tan?mlan?r. Korelasyon e?i?i mevcut projedeki gibi `0.40` olarak korunmu?tur.

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import community as community_louvain

plt.rcParams["figure.dpi"] = 120
sns.set_theme(style="whitegrid")

BASE_DIR = Path.cwd()
RAW_PATH = BASE_DIR / "data" / "raw" / "financial_prices_2014_2024.csv"
RETURNS_PATH = BASE_DIR / "data" / "processed" / "returns_2014_2024.csv"
PRE_CRISIS_PATH = BASE_DIR / "data" / "processed" / "returns_pre_crisis.csv"
POST_CRISIS_PATH = BASE_DIR / "data" / "processed" / "returns_post_crisis.csv"
CORR_PATH = BASE_DIR / "outputs" / "metrics" / "correlation_matrix.csv"
EDGE_PATH = BASE_DIR / "data" / "edges" / "network_edges.csv"

GRAPH_DIR = BASE_DIR / "outputs" / "graphs"
HEATMAP_DIR = BASE_DIR / "outputs" / "heatmaps"
METRICS_DIR = BASE_DIR / "outputs" / "metrics"

for directory in [RAW_PATH.parent, RETURNS_PATH.parent, EDGE_PATH.parent, GRAPH_DIR, HEATMAP_DIR, METRICS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

THRESHOLD = 0.40
START_DATE = "2014-01-01"
END_DATE = "2024-12-31"

## 2. Finansal Varl?klar ve Veri Kayna??

Veriler Yahoo Finance ?zerinden `yfinance` ile toplanm??t?r. Veri seti haz?r bir a? veri seti de?ildir; fiyat verileri ?ekildikten sonra getiri serileri, korelasyon matrisi ve a? kenar listesi proje kapsam?nda ?retilmi?tir.

In [ ]:
TICKERS = {
    "Gold": "GC=F",
    "Silver": "SI=F",
    "Brent_Oil": "BZ=F",
    "WTI_Oil": "CL=F",
    "Natural_Gas": "NG=F",
    "SP500": "^GSPC",
    "NASDAQ": "^IXIC",
    "Dow_Jones": "^DJI",
    "DAX": "^GDAXI",
    "FTSE100": "^FTSE",
    "Nikkei225": "^N225",
    "BIST100": "XU100.IS",
    "USD_TRY": "TRY=X",
    "EUR_TRY": "EURTRY=X",
    "Bitcoin": "BTC-USD",
    "VIX": "^VIX",
    "US_10Y_Bond": "^TNX",
    "Dollar_Index": "DX-Y.NYB",
}

assets = list(TICKERS.keys())
pd.DataFrame({"Asset": TICKERS.keys(), "Yahoo_Finance_Code": TICKERS.values()})

## 3. Ham Verinin Okunmas? veya ?ndirilmesi

Mevcut proje ??kt?lar? korunur. Ham veri dosyas? varsa tekrar indirme yap?lmadan okunur; dosya yoksa ayn? varl?k listesiyle Yahoo Finance ?zerinden veri ?ekilir.

In [ ]:
def download_price_data():
    import yfinance as yf

    price_data = pd.DataFrame()
    for asset_name, ticker in TICKERS.items():
        data = yf.download(ticker, start=START_DATE, end=END_DATE, progress=False)
        if data.empty:
            print(f"Uyar?: {asset_name} i?in veri gelmedi.")
            continue
        if "Adj Close" in data.columns:
            price_data[asset_name] = data["Adj Close"]
        elif "Close" in data.columns:
            price_data[asset_name] = data["Close"]
        else:
            print(f"Uyar?: {asset_name} i?in Close/Adj Close kolonu bulunamad?.")
    price_data.to_csv(RAW_PATH)
    return price_data

if RAW_PATH.exists():
    prices = pd.read_csv(RAW_PATH, index_col="Date", parse_dates=True)
else:
    prices = download_price_data()

prices.head()

## 4. Veri ?n ??leme

Fiyat seviyeleri yerine g?nl?k y?zde getiriler kullan?l?r. Farkl? piyasa takvimlerinden kaynaklanan eksik de?erler ortak i?lem g?nleri ?zerinden temizlenir.

In [ ]:
returns = prices.pct_change().dropna()
returns.to_csv(RETURNS_PATH)
returns.loc["2014":"2019"].to_csv(PRE_CRISIS_PATH)
returns.loc["2020":"2024"].to_csv(POST_CRISIS_PATH)

summary = pd.DataFrame({
    "Metric": ["Start date", "End date", "Observation count", "Asset count"],
    "Value": [returns.index.min().date(), returns.index.max().date(), len(returns), returns.shape[1]],
})
summary

## 5. Korelasyon Matrisi

Finansal varl?k ?iftleri aras?ndaki do?rusal ili?ki Pearson korelasyon katsay?s? ile hesaplan?r. A? kenarlar? olu?turulurken korelasyonun mutlak de?eri dikkate al?n?r; i?aret bilgisi ise pozitif/negatif ili?ki yorumu i?in korunur.

In [ ]:
correlation_matrix = returns.corr(method="pearson")
correlation_matrix.to_csv(CORR_PATH)

plt.figure(figsize=(14, 10))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f", square=True)
plt.title("Financial Asset Correlation Heatmap")
plt.tight_layout()
plt.savefig(HEATMAP_DIR / "correlation_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. A? Modelleme

A? `G = (V, E)` bi?iminde tan?mlan?r. `V` finansal varl?klar?, `E` ise `|correlation| >= 0.40` ko?ulunu sa?layan g??l? ili?kileri temsil eder. A? y?ns?z ve a??rl?kl?d?r.

In [ ]:
G = nx.Graph()
G.add_nodes_from(assets)

for i, asset_1 in enumerate(correlation_matrix.columns):
    for asset_2 in correlation_matrix.columns[i + 1:]:
        corr_value = correlation_matrix.loc[asset_1, asset_2]
        if abs(corr_value) >= THRESHOLD:
            G.add_edge(asset_1, asset_2, weight=float(corr_value))

edge_df = nx.to_pandas_edgelist(G)
edge_df.to_csv(EDGE_PATH, index=False)

print(f"Node count: {G.number_of_nodes()}")
print(f"Edge count: {G.number_of_edges()}")
edge_df.head(10)

## 7. Kom?uluk Matrisi ve Aktif Alt A?

E?ik de?erinden sonra baz? varl?klar izole kalabilir. Bu nedenle raporda hem 18 d???ml? tam a? hem de yaln?zca kenar? olan aktif alt a? ayr? ayr? yorumlan?r.

In [ ]:
adjacency_matrix = nx.to_pandas_adjacency(G, weight="weight")
active_nodes = [node for node, degree in G.degree() if degree > 0]
active_G = G.subgraph(active_nodes).copy()

print(f"Active node count: {active_G.number_of_nodes()}")
print(f"Active edge count: {active_G.number_of_edges()}")
adjacency_matrix.round(2)

## 8. Temel A? ?l??tleri

Bu b?l?mde d???m say?s?, kenar say?s?, yo?unluk, ortalama derece, bile?en say?s?, en b?y?k bile?en ?ap? ve ortalama k?meleme katsay?s? hesaplan?r.

In [ ]:
def graph_summary(graph, label):
    components = list(nx.connected_components(graph))
    largest_component = graph.subgraph(max(components, key=len)).copy() if components else nx.Graph()
    diameter = nx.diameter(largest_component) if largest_component.number_of_nodes() > 1 else 0
    return {
        "Network": label,
        "Nodes": graph.number_of_nodes(),
        "Edges": graph.number_of_edges(),
        "Density": nx.density(graph),
        "Average degree": (2 * graph.number_of_edges() / graph.number_of_nodes()) if graph.number_of_nodes() else 0,
        "Connected components": len(components),
        "Largest component size": largest_component.number_of_nodes(),
        "Largest component diameter": diameter,
        "Average clustering": nx.average_clustering(graph) if graph.number_of_nodes() else 0,
    }

summary_df = pd.DataFrame([
    graph_summary(G, "Full network including isolated nodes"),
    graph_summary(active_G, "Active network excluding isolated nodes"),
])
summary_df.to_csv(METRICS_DIR / "global_network_metrics.csv", index=False)
summary_df

## 9. A? G?rselle?tirmesi

D???m boyutu ve rengi degree centrality de?erini, kenar kal?nl??? korelasyon g?c?n? g?sterir. Pozitif ili?kiler k?rm?z?, negatif ili?kiler mavi ?izilmi?tir.

In [ ]:
pos = nx.spring_layout(G, seed=42, k=1.3, iterations=100)
degree_centrality = nx.degree_centrality(G)

node_sizes = [degree_centrality[node] * 9000 + 1000 for node in G.nodes()]
node_colors = [degree_centrality[node] for node in G.nodes()]
positive_edges = [(u, v) for u, v in G.edges() if G[u][v]["weight"] > 0]
negative_edges = [(u, v) for u, v in G.edges() if G[u][v]["weight"] < 0]

plt.figure(figsize=(16, 11))
nodes = nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=node_colors, cmap=plt.cm.plasma, edgecolors="black", linewidths=1.2, alpha=0.92)
nx.draw_networkx_edges(G, pos, edgelist=positive_edges, width=[abs(G[u][v]["weight"]) * 7 for u, v in positive_edges], edge_color="red", alpha=0.6, label="Pozitif korelasyon")
nx.draw_networkx_edges(G, pos, edgelist=negative_edges, width=[abs(G[u][v]["weight"]) * 7 for u, v in negative_edges], edge_color="blue", alpha=0.6, label="Negatif korelasyon")
nx.draw_networkx_labels(G, pos, font_size=8, font_weight="bold")
plt.colorbar(nodes, label="Degree Centrality")
plt.legend(loc="upper right")
plt.title("Financial Asset Correlation Network")
plt.axis("off")
plt.tight_layout()
plt.savefig(GRAPH_DIR / "network_graph_enhanced.png", dpi=300, bbox_inches="tight")
plt.show()

## 10. Merkezilik Analizi

Hocan?n ?ablonunda istenen d?rt merkezilik ?l??t? hesaplan?r: degree, betweenness, closeness ve eigenvector centrality. Her ?l??te g?re ilk 5 d???m ayr? ayr? g?r?lebilir.

In [ ]:
centrality_df = pd.DataFrame({
    "Degree_Centrality": pd.Series(nx.degree_centrality(G)),
    "Betweenness_Centrality": pd.Series(nx.betweenness_centrality(G, weight="weight")),
    "Closeness_Centrality": pd.Series(nx.closeness_centrality(G)),
    "Eigenvector_Centrality": pd.Series(nx.eigenvector_centrality(G, max_iter=1000)),
    "Clustering_Coefficient": pd.Series(nx.clustering(G)),
})
centrality_df.to_csv(METRICS_DIR / "network_metrics.csv")
centrality_df.sort_values("Degree_Centrality", ascending=False).head(10)

In [ ]:
top5_tables = {}
for column in ["Degree_Centrality", "Betweenness_Centrality", "Closeness_Centrality", "Eigenvector_Centrality"]:
    top5_tables[column] = centrality_df.sort_values(column, ascending=False).head(5)[[column]]
    print(f"
Top 5 - {column}")
    display(top5_tables[column])

## 11. Derece Da??l?m?

Bu grafik, a?da hub benzeri d???mlerin olup olmad???n? g?rmek i?in derece de?erlerinin da??l?m?n? g?sterir.

In [ ]:
degree_series = pd.Series(dict(G.degree()), name="Degree").sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=degree_series.index, y=degree_series.values, color="#4C78A8")
plt.title("Degree Distribution by Asset")
plt.ylabel("Degree")
plt.xlabel("Asset")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(GRAPH_DIR / "degree_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

degree_series.to_frame()

## 12. Topluluk Analizi

Louvain y?ntemi aktif alt a? ?zerinde uygulan?r. Negatif korelasyonlar i?in algoritmada a??rl?k olarak mutlak korelasyon kullan?l?r; ili?kinin y?n? ayr? bir ?zellik olarak korunur.

In [ ]:
community_G = nx.Graph()
for u, v, data in G.edges(data=True):
    corr = data["weight"]
    community_G.add_edge(u, v, weight=abs(corr), correlation=corr, relation_type="positive" if corr > 0 else "negative")

partition = community_louvain.best_partition(community_G, weight="weight", random_state=42)
modularity = community_louvain.modularity(partition, community_G, weight="weight")

community_df = pd.DataFrame({"Node": list(partition.keys()), "Community": list(partition.values())}).sort_values(["Community", "Node"])
community_df.to_csv(METRICS_DIR / "community_results.csv", index=False)

print(f"Community count: {community_df['Community'].nunique()}")
print(f"Modularity: {modularity:.4f}")
community_df

## 13. Topluluk G?rselle?tirmesi

D???m renkleri Louvain algoritmas?n?n buldu?u topluluklar? temsil eder. Bu g?rsel finansal varl?klar?n hangi alt piyasa gruplar? etraf?nda k?melendi?ini okumay? kolayla?t?r?r.

In [ ]:
pos_comm = nx.spring_layout(community_G, seed=42, k=1.3)
communities = sorted(set(partition.values()))
color_map = plt.cm.Set3
node_colors = [color_map(communities.index(partition[node]) / max(len(communities), 1)) for node in community_G.nodes()]
node_sizes = [nx.degree_centrality(community_G)[node] * 9000 + 1000 for node in community_G.nodes()]

plt.figure(figsize=(14, 10))
nx.draw_networkx_nodes(community_G, pos_comm, node_size=node_sizes, node_color=node_colors, edgecolors="black", linewidths=1.2, alpha=0.92)
nx.draw_networkx_edges(community_G, pos_comm, edgelist=[(u, v) for u, v in community_G.edges() if community_G[u][v]["correlation"] > 0], edge_color="red", alpha=0.6)
nx.draw_networkx_edges(community_G, pos_comm, edgelist=[(u, v) for u, v in community_G.edges() if community_G[u][v]["correlation"] < 0], edge_color="blue", alpha=0.6)
nx.draw_networkx_labels(community_G, pos_comm, font_size=9, font_weight="bold")
plt.title("Financial Asset Community Network")
plt.axis("off")
plt.tight_layout()
plt.savefig(GRAPH_DIR / "community_network.png", dpi=300, bbox_inches="tight")
plt.show()

## 14. K?sa Dayan?kl?l?k Analizi

?ablondaki dayan?kl?l?k beklentisi i?in iki hedefli ??karma senaryosu uygulan?r: en y?ksek degree centrality d???m?n?n ??kar?lmas? ve en y?ksek betweenness centrality d???m?n?n ??kar?lmas?. Sonu?lar a??n kenar say?s?, yo?unluk, bile?en say?s? ve en b?y?k bile?en b?y?kl??? ?zerinden kar??la?t?r?l?r.

In [ ]:
def robustness_summary(graph, scenario, removed_node="-"):
    components = list(nx.connected_components(graph))
    return {
        "Scenario": scenario,
        "Removed node": removed_node,
        "Nodes": graph.number_of_nodes(),
        "Edges": graph.number_of_edges(),
        "Density": nx.density(graph),
        "Connected components": len(components),
        "Largest component size": max((len(component) for component in components), default=0),
        "Average clustering": nx.average_clustering(graph) if graph.number_of_nodes() else 0,
    }

top_degree_node = centrality_df["Degree_Centrality"].idxmax()
top_betweenness_node = centrality_df["Betweenness_Centrality"].idxmax()

robustness_rows = [robustness_summary(G, "Original network")]
for scenario, node in [
    ("Remove highest degree node", top_degree_node),
    ("Remove highest betweenness node", top_betweenness_node),
]:
    H = G.copy()
    H.remove_node(node)
    robustness_rows.append(robustness_summary(H, scenario, node))

robustness_df = pd.DataFrame(robustness_rows)
robustness_df.to_csv(METRICS_DIR / "robustness_analysis.csv", index=False)
robustness_df

## 15. K?sa Sonu?

Analiz, k?resel hisse senedi endeksleri ve VIX etraf?nda g??l? bir ?ekirdek yap? olu?tu?unu; de?erli metaller, d?viz kurlar? ve enerji varl?klar?n?n ise ayr? k???k segmentler halinde k?melendi?ini g?stermektedir. ?zole kalan baz? varl?klar, se?ilen korelasyon e?i?ine g?re ana ili?ki ?ekirde?inin d???nda kalm??t?r. Bu nedenle a? hem merkezde yo?unla?an bir piyasa ?ekirde?i hem de segmentlere ayr?lan finansal alt gruplar i?ermektedir.